In [ ]:
print("hello world!")

In [ ]:
import os
import re
import sys
import json
import torch
import pickle
import contextlib
import numpy as np
from tqdm import tqdm
from typing import List, Tuple, Dict

from qiskit_aer import AerSimulator
from pytket.extensions.qiskit.backends.aer import AerBackend
# from qiskit.providers.aer import AerSimulator
# from pytket.extensions.qiskit import AerBackend


from lambeq.backend.grammar import Diagram, Id
from lambeq import (
    AtomicType,
    IQPAnsatz,
    RemoveCupsRewriter,
    SimpleRewriteRule,
    Rewriter,
    UnifyCodomainRewriter,
    DepCCGParser
)

# import depccg
# from lambeq import ( CCGParser, CCGTree, CCGRuleUseError, CCGRule, CCGType,
#                     CCGBankParseError, CCGBankParser, DepCCGParseError )


In [ ]:
import logging
logging.getLogger("allennlp").setLevel(logging.WARNING)
logging.getLogger("depccg").setLevel(logging.WARNING)
parser = DepCCGParser(model='elmo', device=-1) # device=:  -1 == CPU | 0 == GPU | 1 == second GPU

In [ ]:
_CLEAN_RE = re.compile(r"[^\w\s']")
# parser    = DepCCGParser(model='elmo', device=0) # device=:  -1 == CPU | 0 == GPU | 1 == second GPU

ansatz    = IQPAnsatz(
    {AtomicType.SENTENCE: 1,
     AtomicType.NOUN:     1,
     AtomicType.PREPOSITIONAL_PHRASE: 0},
    n_layers=2, n_single_qubit_params=3
)

def create_rewriter():
    # Rule to delete conjunction boxes (“and”, “but”) # just the wire, no box
    conj_rule = SimpleRewriteRule(cod=AtomicType.CONJUNCTION, template=Id(AtomicType.CONJUNCTION))

    # Rule to delete punctuation boxes (commas, quotes, dashes)
    punc_rule = SimpleRewriteRule(cod=AtomicType.PUNCTUATION, template=Id(AtomicType.PUNCTUATION))

    # remove_pp2 = SimpleRewriteRule(cod=AtomicType.PREPOSITIONAL_PHRASE, template=Id(AtomicType.SENTENCE))
    remove_pp = SimpleRewriteRule(cod=AtomicType.PREPOSITIONAL_PHRASE, template=Id(AtomicType.NOUN))

    rewriter = Rewriter(
        [
            'coordination', 'determiner',
            'postadverb', 'preadverb',
            'connector', 'auxiliary',
            'prepositional_phrase',
            'subject_rel_pronoun',
            'object_rel_pronoun',
        ]
    )
    rewriter.add_rules(remove_pp, punc_rule, conj_rule)
    return rewriter

rewriter = create_rewriter()
remove_cups = RemoveCupsRewriter()
unify = UnifyCodomainRewriter(output_type=AtomicType.SENTENCE)

In [ ]:
#################################################
#### Mainly to test if GPU works with:       ####
#### sim.run(tk_circ, n_shots=1) and         ####
#### backend.run_circuit(tk_circ, n_shots=1) ####
#################################################


#### ner kartais cia breikalo skaitoma po viena eilute, istraukiami elemntai  ir lygiai taip pat idedami i `data` kintamji??
def load_cnn_extractive(file_path: str, amount: int = None):
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if amount and i == amount:
                break
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue

            text = record.get("text")
            summary = record.get("summary")
            text_sentences = record.get("text_sentences")
            summary_sentences = record.get("summary_sentences")
            labels = record.get("labels")
            if text is not None or summary is not None:
                data.append(
                    {
                        "text": text,
                        "summary": summary,
                        "text_sentences": text_sentences,
                        "summary_sentences": summary_sentences,
                        "labels": labels,
                    }
                )
            else:
                print(f"{i}. NULL INSTANCE\ntext: {text}\n{summary}\n")
    return data


def load_qe():
    with open("saves/qe.pkl", "rb") as f:
        loaded_data = pickle.load(f)

    dste = loaded_data["dste"]
    length = 0
    for d in dste:
        length += len(d["circuits"])

    print(type(dste[0]["circuits"][0]))
    print(f"Successfully Loaded {length} circuits!")
    return dste


def load_PreSumm_pts(left=0, right=144, ds_purpose = "train"):
    raw_ds = []
    k = left
    
    while k < right:
        file_path = f"Dataset/Raw/cnn_dailymail/_PreSumm/cnndm.{ds_purpose}.{k}.bert.pt"
        loaded_data = torch.load(file_path)

        for i, line in enumerate(loaded_data):

            quantum_state_distribution_labels = []
            for label in line["src_sent_labels"]:
                if label:
                    quantum_state_distribution_labels.append([0,1])
                else:
                    quantum_state_distribution_labels.append([1,0])

            raw_ds.append(
                {
                    "text_sentences": line["src_txt"],
                    # "org_labels": line["src_sent_labels"],
                    "labels": quantum_state_distribution_labels
                }
            )
        k += 1
    return raw_ds



In [ ]:
# def testing_lambeq_DepCCG(list_of_sentences: List):
#     for sentence in list_of_sentences:

def preprocess_and_encode_DepCCG(dataset):
    encoded_data, errors = [], []
    errs1, errs2, errs3, errs4 = [], [], [], []

    for i, ds_dict in enumerate(tqdm(dataset, desc="Filtering and Encoding dataset")):
    # for i, ds_dict in enumerate(dataset):
        with open(os.devnull, 'w') as devnull, \
         contextlib.redirect_stdout(devnull), \
         contextlib.redirect_stderr(devnull):

            text_sentences = ds_dict['text_sentences']
            labels         = ds_dict['labels']

            sentences_simplified = sentence_simplify(text_sentences)
            diagrams, remove     = sent2diagrams(sentences_simplified)
            diagrams             = remove_by_idx(diagrams, remove)
            text_sentences       = remove_by_idx(text_sentences, remove)
            labels               = remove_by_idx(labels, remove)

            normalized_diagrams, remove, errs2 = normalize(diagrams)
            text_sentences                     = remove_by_idx(text_sentences, remove)
            labels                             = remove_by_idx(labels, remove)

            circuits, remove, errs3 = quantum_encode(normalized_diagrams)
            text_sentences          = remove_by_idx(text_sentences, remove)
            labels                  = remove_by_idx(labels, remove)

            # circuits, remove, errs4 = will_train(circuits)
            # text_sentences          = remove_by_idx(text_sentences, remove)
            # labels                  = remove_by_idx(labels, remove)

            encoded_data.append({'circuits': circuits, 'labels': labels, 'original_text_sentences': text_sentences})


            errors += errs1 + errs2 + errs3 + errs4
            print("text_sentences:", text_sentences)
            print("sentences_simplified:", len(sentences_simplified), sentences_simplified)
            print("noramlized_sentences:", len(normalized_diagrams), normalized_diagrams)
            print("circuits:", circuits)
            # print()

    return encoded_data, errors
        

In [ ]:
dataset = load_cnn_extractive(file_path="Dataset/Raw/cnn_dailymail/v1/train.jsonl",amount=10)
# dataset = load_PreSumm_pts(0,1,'test')

In [ ]:
encoded_data4, errors4 = preprocess_and_encode_DepCCG(dataset)


In [ ]:
encoded_data3, errors3 = preprocess_and_encode_DepCCG(dataset)


In [ ]:
encoded_data2, errors2 = preprocess_and_encode_DepCCG(dataset)


In [ ]:
encoded_data, errors = preprocess_and_encode_DepCCG(dataset)
# on CPU it takes ~5m 56.5s to necode first 10 articles
# on `GPU` it takes ~4m 57.5s to necode first 10 articles

# on CPU and feeding whole articles (not single sentences) it takes ~3m 4.2s to necode first 10 articles
# on `GPU` and feeding whole articles (not single sentences) it takes ~2m 44.2s to necode first 10 articles

# on CPU and feeding multiple (3) articles it takes ~_m _._s to necode first 10 articles
# on `GPU` and feeding multiple (3) articles it takes ~_m _._s to necode first 10 articles

In [ ]:
find_mismatches(encoded_data4, encoded_data2)

print(encoded_data[7]["circuits"][11])
print(encoded_data3[7]["circuits"][11])

In [ ]:
print(errors)
print(errors2)
print(errors3)

# print(encoded_data[0]["circuits"][0])

In [ ]:
with open('Dataset/Encoded/cnn_dailymail/cnn_dailymail_train_0_9_somehowDifferent.pkl', 'wb') as file:
    pickle.dump(encoded_data2, file)

In [ ]:
with open('Dataset/Encoded/cnn_dailymail/cnn_dailymail_train_0_9.pkl', 'rb') as file:
    loaded_data = pickle.load(file)

In [ ]:
def get_deep_type(obj):
    if isinstance(obj, list):
        # We look at the unique types inside the list to keep it readable
        inner_types = {get_deep_type(item) for item in obj}
        return f"List[{' | '.join(sorted(inner_types))}]"

    elif isinstance(obj, dict):
        # We summarize the types of all keys and all values
        key_types = {get_deep_type(k) for k in obj.keys()}
        val_types = {get_deep_type(v) for v in obj.values()}
        return f"Dict[{' | '.join(sorted(key_types))}, {' | '.join(sorted(val_types))}]"

    else:
        # Return the class name (e.g., 'Diagram' or 'str')
        return type(obj).__name__

def find_mismatches(data_a, data_b):
    # 1. Check if the outer lists are even the same length
    if len(data_a) != len(data_b):
        print(
            f"❌ [CRITICAL] Outer List Length Mismatch: List A={len(data_a)}, List B={len(data_b)}"
        )

    # Iterate through the top-level list
    for i, (dict_a, dict_b) in enumerate(zip(data_a, data_b)):
        # Check if the keys in the dictionaries match
        keys_a = set(dict_a.keys())
        keys_b = set(dict_b.keys())

        if keys_a != keys_b:
            print(f"❌ [Index {i}] Key Mismatch:")
            print(f"   Keys only in A: {keys_a - keys_b}")
            print(f"   Keys only in B: {keys_b - keys_a}")
            continue  # Skip to next list item if keys don't match

        # Check values for each key
        for key in keys_a:
            list_a = dict_a[key]
            list_b = dict_b[key]

            # 2. Check lengths of the lists inside the dictionary
            if len(list_a) != len(list_b):
                print(f"❌ [Index {i}][Key: '{key}'] Inner List Length Mismatch:")
                print(f"   Length A: {len(list_a)}")
                print(f"   Length B: {len(list_b)}")

            # 3. Check individual elements inside those lists
            # This handles List[Diagram], List[List[int]], and List[str]
            for j, (val_a, val_b) in enumerate(zip(list_a, list_b)):
                if val_a != val_b:
                    print(
                        f"❌ [Index {i}][Key: '{key}'][Inner Index {j}] Content Mismatch!"
                    )
                    print(f"   Type A: {type(val_a).__name__}")
                    print(f"   Type B: {type(val_b).__name__}")

                    # If they are small (like List[int] or str), print the actual value
                    if not hasattr(
                        val_a, "draw"
                    ):  # Don't print full Diagrams, they are too big
                        print(f"   Value A: {val_a}")
                        print(f"   Value B: {val_b}")
                    else:
                        print(
                            f"   (Diagram content differs - possibly different boxes or wires)"
                        )

In [ ]:
simulator = AerSimulator(
    method="statevector", device="GPU",
    precision="single",         # 32-bit float for ~2× speedup on large statevectors
    cuStateVec_enable=True,     # turn on NVIDIA cuStateVec kernels
    batched_shots_gpu=True,     # batch thousands of shots very efficiently on GPU
    batched_shots_gpu_max_qubits=40,
    num_threads_per_device=2    # limit CPU threads per GPU to reduce overhead
)

backend = AerBackend(noise_model=None, simulation_method="statevector")
backend._qiskit_backend = simulator

comp_pass = backend.default_compilation_pass(2)

In [ ]:
#############################################
# Step 2. Preprocessing and lambeq Pipeline
#############################################

def remove_by_idx(ls, remove):
    if not remove:
        return ls
    remove.sort(reverse=True)
    for n in remove:
        ls.pop(n)
    return ls

def sentence_simplify(sentences):
    return [_CLEAN_RE.sub("", s) for s in sentences if s is not None]


def sent2diagrams(sentences):
    none_idx = []
    diagrams = parser.sentences2diagrams(sentences, tokenised=False, suppress_exceptions=True)
    for i, diag in enumerate(diagrams):
        if diag is None:
            none_idx.append(i)
    
    print(f"Whilst parsing sentences2diagrams, lost {len(none_idx)} due to Null, out of {len(sentences)}.")

    return diagrams, none_idx


def normalize(sentence_diagrams):
    diagrams_normalized, none_idx, errs = [], [], []
    drop_rewrite = 0
    drop_cups = 0
    for i, d in enumerate(sentence_diagrams):
        try:
            d = rewriter(d)
            if (d is None):
                none_idx.append(i)
                drop_rewrite += 1
                continue

            d = remove_cups(d)
            d = d.normal_form()
            d = d.pregroup_normal_form()
            d = unify(d)

        except Exception as e:
            none_idx.append(i)
            errs.append(f"{e} ( normalize() )")
            drop_cups += 1
            continue

        diagrams_normalized.append(d)

    print(f"Dropped {drop_rewrite} diagrams in rewrite, {drop_cups} in cup removal, out of {len(sentence_diagrams)}.")

    return diagrams_normalized, none_idx, errs

def quantum_encode(diagrams: "List"):
    encoded_diagrams, remove, errs = [], [], []
    for i, diagram in enumerate(diagrams):
        try:
            circ    = ansatz(diagram)
            # circ = circ.to_tk()
            encoded_diagrams.append(circ)
        except Exception as e:
            errs.append(f"{e} ( quantum_encode() )")
            remove.append(i)

    return encoded_diagrams, remove, errs

def will_train(
    circuits,
    qubit_limit: int = 40,
    mem_limit_bytes: int = 7 * 2**30,   # 7 GiB
):

    valid, invalid_idxs, errs = [], [], []
    for idx, circ in enumerate(circuits):
        try:
            tk_circ = circ.to_tk()
            needed = 16 * (2 ** tk_circ.n_qubits)
            if needed > mem_limit_bytes:
                raise RuntimeError(f"Needs {needed} bytes > limit {mem_limit_bytes} bytes ({needed/2**20:.0f} MiB > {(mem_limit_bytes/2**20):.0f} MiB). ")

            comp_pass.apply(tk_circ)

            syms = tk_circ.free_symbols()
            if syms:
                bind_map = {s: 0.0 for s in syms}
                tk_circ.symbol_substitution(bind_map)

            simulator.run(tk_circ, n_shots=1)
            # backend.run_circuit(tk_circ, n_shots=1)

        except Exception as e:
            invalid_idxs.append(idx)
            errs.append(f"{e} ( will_train() )")
            continue

        valid.append( circ )

    return valid, invalid_idxs, errs



def preprocess_and_encode(dataset):
    encoded_data, errors = [], []
    errs1, errs2, errs3, errs4 = [], [], [], []

    for i, ds_dict in enumerate(tqdm(dataset, desc="Filtering and Encoding dataset")):
        # with open(os.devnull, 'w') as devnull, \
        #  contextlib.redirect_stdout(devnull), \
        #  contextlib.redirect_stderr(devnull):

        text_sentences = ds_dict['text_sentences']
        labels         = ds_dict['labels']

        sentences_simplified = sentence_simplify(text_sentences)
        diagrams, remove     = sent2diagrams(sentences_simplified)
        diagrams             = remove_by_idx(diagrams, remove)
        text_sentences       = remove_by_idx(text_sentences, remove)
        labels               = remove_by_idx(labels, remove)

        normalized_diagrams, remove, errs2 = normalize(diagrams)
        text_sentences                     = remove_by_idx(text_sentences, remove)
        labels                             = remove_by_idx(labels, remove)

        circuits, remove, errs3 = quantum_encode(normalized_diagrams)
        text_sentences          = remove_by_idx(text_sentences, remove)
        labels                  = remove_by_idx(labels, remove)

        circuits, remove, errs4 = will_train(circuits)
        text_sentences          = remove_by_idx(text_sentences, remove)
        labels                  = remove_by_idx(labels, remove)

        encoded_data.append({'circuits': circuits, 'labels': labels, 'original_text_sentences': text_sentences})


        errors += errs1 + errs2 + errs3 + errs4
        print("text_sentences:", text_sentences)
        print("sentences_simplified:", len(sentences_simplified), sentences_simplified)
        print("noramlized_sentences:", len(normalized_diagrams), normalized_diagrams)
        print("circuits:", circuits)
        # print()

    return encoded_data, errors

In [ ]:
def brbrpatapim(filename, left, right):
    dataset    = load_cnn_extractive(filename)
    dst        = dataset[left:right]
    dste, errs = preprocess_and_encode(dst)
    print("\nDone!")
    
    return dste, errs

train, errs_train = brbrpatapim("Dataset/Raw/cnn_dailymail/train.jsonl", 35, 40)